In [27]:
import geopandas as gpd
import pandas as pd
import os
import warnings


# ---- CONFIG ----
# Path to your input CSV file

input_shp = r"Z:\data\im-nca-colombia\mec versión 2.1\mec_original\fixed_projection\e_eccmc_ver21_100k_3116_old.shp"
# input_shp = r"Y:\z_resources\im-nca-colombia\2025\mapa_ecosistemas_continentales_costeros_marinos_100k_2024\shape\e_eccmc_100k_2024_magna.shp"
# Path to your output CSV file
output_csv = "unique_values.csv"


In [28]:
# ---- LOAD SHAPEFILE ----
gdf = gpd.read_file(input_shp)

In [30]:
# ---- GET UNIQUE VALUES ----
unique_values = {}
for col in gdf.columns:
    print(f"we are in column: {col}")
    # Skip geometry column if you don't want it
    if col != gdf.geometry.name:
        unique_values[col] = gdf[col].dropna().unique().tolist()

# ---- ALIGN TO SAME LENGTH ----
max_len = max(len(v) for v in unique_values.values())
for col in unique_values:
    unique_values[col] += [""] * (max_len - len(unique_values[col]))

# ---- CREATE NEW DATAFRAME ----
unique_df = pd.DataFrame(unique_values)

# ---- SAVE TO CSV ----
unique_df.to_csv(output_csv, index=False)

print(f"✅ Unique values saved to: {output_csv}")

we are in column: OBJECTID
we are in column: TIPO_ECOSI
we are in column: GRADO_TRAN
we are in column: GRAN_BIOMA
we are in column: BIOMA_PREL
we are in column: BIOMA_IAvH
we are in column: ECOS_SINTE
we are in column: ECOS_GENER
we are in column: UNIDAD_SIN
we are in column: AMBIENTE_A
we are in column: SUBSISTEMA
we are in column: ZONA_HIDRO
we are in column: ORIGEN
we are in column: TIPO_AGUA
we are in column: CLIMA
we are in column: PAISAJE
we are in column: RELIEVE
we are in column: SUELOS
we are in column: AMB_EDAFOG
we are in column: DESC_AMB_E
we are in column: COBERTURA
we are in column: SUSTRATO
we are in column: ZONA
we are in column: TEMPERATUR
we are in column: SALINIDAD
we are in column: PROVINCIA
we are in column: ECO_REGION
we are in column: ECO_ZONA
we are in column: ORIGEN_MAR
we are in column: CONFIGURAC
we are in column: CLAS_BIOTI
we are in column: SUBCLAS_BI
we are in column: GRUPO_BIOT
we are in column: SECTORES
we are in column: Area_ha
we are in column: UNI_BIO

In [3]:
# 1️⃣ Filter rows where 'eco_g_code' is empty or null
empty_eco_g_code = gdf[gdf['eco_g_code'].isna() | (gdf['eco_g_code'] == '')]

# 2️⃣ Get the unique values from 'ecos_gener' column
unique_ecos_gener = empty_eco_g_code['ecos_gener'].dropna().unique()

In [4]:
unique_ecos_gener

array(['Rio de Aguas Blancas', 'Vegetación Secundaria',
       'Transicional Transformado Costero',
       'Bosque Ripario Inundable Subandino', 'Agroecosistema Cañero',
       'Rio de Aguas Claras', 'Laguna Costera',
       'Pradera de Pastos Marinos'], dtype=object)

In [25]:
"""Get paris unique values"""
def get_unique_matching_pairs_merged(gdf, column_pairs, match_equal=False):
    """
    Extract unique value pairs from multiple column pairs in a GeoDataFrame,
    merge them into a single DataFrame, and print logs of progress.

    Parameters
    ----------
    gdf : GeoDataFrame
        Input GeoDataFrame.
    column_pairs : dict
        Mapping of column1 -> column2 for which to extract pairs.
    match_equal : bool, optional
        If True, only include rows where col1 == col2.

    Returns
    -------
    GeoDataFrame
        A merged DataFrame with unique pairs for each valid column pair.
    """
    pair_dfs = []
    missing_pairs = []

    print("Starting processing of column pairs...\n")

    for col1, col2 in column_pairs.items():
        # Validate columns exist
        if col1 not in gdf.columns or col2 not in gdf.columns:
            missing_pairs.append((col1, col2))
            print(f"⚠️ Column pair ({col1}, {col2}) not found in GeoDataFrame. Skipping.")
            continue

        print(f"Processing pair: ({col1}, {col2})")

        temp = gdf[[col1, col2]].copy()

        # Convert to string to avoid mixed-type issues
        temp[col1] = temp[col1].astype(str)
        temp[col2] = temp[col2].astype(str)

        # Filter for equal pairs if requested
        if match_equal:
            temp = temp[temp[col1] == temp[col2]]

        # Drop duplicates
        temp = temp.drop_duplicates().reset_index(drop=True)

        print(f"  ✅ Found {len(temp)} unique pairs for ({col1}, {col2})\n")

        pair_dfs.append(temp)

    # Warn if missing columns
    if missing_pairs:
        missing_str = ", ".join([f"({a}, {b})" for a, b in missing_pairs])
        warnings.warn(
            f"The following column pairs were not found in the GeoDataFrame: {missing_str}",
            UserWarning
        )

    if not pair_dfs:
        print("No valid column pairs found. Returning empty GeoDataFrame.")
        return gpd.GeoDataFrame()

    # Merge all valid pairs side by side
    merged = pd.concat(pair_dfs, axis=1)
    merged = merged.loc[:, ~merged.columns.duplicated()]

    print("All pairs processed. Returning merged DataFrame.\n")
    return merged


mec_2_column_pairs = {"clima": "clima_code" , "amb_acuati": "ambi_code", "gra_trans": "grado_code", "bioma_prel": "bioma_code", "tipo_agua": 
                      "agua_code", "suelos": "edafo_code", "u_biotica": "u_bio_code", 
                      "tipo_ecos": "t_eco_code", "ecos_gener": "eco_g_code"} # "amb_edafog": "edafo_code"


mec_1_column_pairs = {"RULEID": "ECOS_GENER", "clima_code": "CLIMA", "ambi_code": "AMBIENTE_A", "grado_code": "GRADO_TRAN", "agua_code": "TIPO_AGUA", "u_bio_code": "UNI_BIOTIC"}

result = get_unique_matching_pairs_merged(gdf, mec_2_column_pairs, match_equal=False)
print(result)

Starting processing of column pairs...

Processing pair: (clima, clima_code)
  ✅ Found 21 unique pairs for (clima, clima_code)

Processing pair: (amb_acuati, ambi_code)
  ✅ Found 4 unique pairs for (amb_acuati, ambi_code)

Processing pair: (gra_trans, grado_code)
  ✅ Found 2 unique pairs for (gra_trans, grado_code)

Processing pair: (bioma_prel, bioma_code)
  ✅ Found 13 unique pairs for (bioma_prel, bioma_code)

Processing pair: (tipo_agua, agua_code)
  ✅ Found 4 unique pairs for (tipo_agua, agua_code)

Processing pair: (suelos, edafo_code)
  ✅ Found 685 unique pairs for (suelos, edafo_code)

Processing pair: (u_biotica, u_bio_code)
  ✅ Found 69 unique pairs for (u_biotica, u_bio_code)

Processing pair: (tipo_ecos, t_eco_code)
  ✅ Found 5 unique pairs for (tipo_ecos, t_eco_code)

Processing pair: (ecos_gener, eco_g_code)
  ✅ Found 87 unique pairs for (ecos_gener, eco_g_code)

All pairs processed. Returning merged DataFrame.

                  clima clima_code    amb_acuati ambi_code   

In [26]:
result.to_csv(output_csv, index=False, encoding='utf-8-sig') # for latin characters